[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/10_numpy_bases.ipynb)



# NumPy : les bases

Une liste Python peut contenir n'importe quoi, dans n'importe quel ordre. Cette
souplesse a un prix : chaque élément est un objet séparé en mémoire, et toute
opération passe par une boucle.

Un tableau NumPy contient un seul type, rangé dans un bloc mémoire continu.
C'est ce qui permet de calculer sur des millions de valeurs sans écrire de
boucle, avec des performances comparables à du C.

C'est la fondation de tout le reste : Pandas, scikit-learn, Matplotlib et
TensorFlow manipulent des tableaux NumPy.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Créer un tableau, et l'erreur du premier appel |
| 2 | `shape` et `ndim` : la forme d'un tableau |
| 3 | Les constructeurs : `zeros`, `ones`, `full`, `eye` |
| 4 | `linspace` et `arange` |
| 5 | Les types de données |
| 6 | L'aléatoire et la reproductibilité |
| 7 | Assembler des tableaux |
| 8 | Changer la forme : `reshape`, `ravel`, `squeeze` |

## Trois pièges annoncés

`(3,)` et `(3, 1)` ne sont pas la même chose, et cette différence provoque la
majorité des erreurs de dimension.

`reshape` ne modifie pas le tableau, il en renvoie un nouveau.

Et `ravel` renvoie une vue alors que `flatten` renvoie une copie les
modifier n'a pas le même effet.

Prérequis : les notebooks 02 à 09.

## 1. Créer un tableau

La convention universelle est d'importer NumPy sous l'alias `np`. Tout le code
que tu liras l'utilise.

In [59]:
import numpy as np

print(np.__version__)

2.0.2


### L'erreur du premier appel

C'est celle que tout le monde fait une fois.

In [60]:
try:
    A = np.array(1, 2, 3)
except TypeError as e:
    print('TypeError :', e)

TypeError : array() takes from 1 to 2 positional arguments but 3 were given


Le message dit que la fonction accepte au plus deux arguments positionnels et
qu'on en a donné trois.

La raison : `np.array` prend une séquence en premier argument, pas une suite
de nombres. Il faut des crochets ou des parenthèses supplémentaires.

In [61]:
A = np.array([1, 2, 3]) # depuis une liste
B = np.array((1, 2, 3)) # depuis un tuple

print(A)
print(B)
print('identiques :', np.array_equal(A, B))
print(type(A))

[1 2 3]
[1 2 3]
identiques : True
<class 'numpy.ndarray'>


Le type est `ndarray`, pour n-dimensional array. C'est la classe centrale de
NumPy celle dont on lisait la documentation au notebook précédent.

La liste ou le tuple sont convertis à la création ; le tableau obtenu n'a plus
rien à voir avec la structure d'origine.

### Deux dimensions

Une liste de listes donne un tableau à deux dimensions, à condition que toutes
les sous-listes aient la même longueur.

In [62]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])

print(M)
print('forme :', M.shape)

[[1 2 3]
 [4 5 6]]
forme : (2, 3)


Si les longueurs diffèrent, NumPy ne peut pas construire de bloc rectangulaire :

In [63]:
try:
    np.array([[1, 2, 3], [4, 5]])
except ValueError as e:
    print('ValueError :', str(e)[:80])

ValueError : setting an array element with a sequence. The requested array has an inhomogeneo


C'est une différence fondamentale avec les listes Python, qui acceptent
n'importe quelle imbrication. Un tableau NumPy est rectangulaire, toujours.

## 2. shape et ndim

`shape` donne la taille de chaque dimension, sous forme de tuple. `ndim` donne
le nombre de dimensions.

In [64]:
A = np.array([1, 2, 3])
M = np.array([[1, 2, 3], [4, 5, 6]])
print('A : shape =', A.shape, ' ndim =', A.ndim, ' size =', A.size)
print('M : shape =', M.shape, ' ndim =', M.ndim, ' size =', M.size)

A : shape = (3,)  ndim = 1  size = 3
M : shape = (2, 3)  ndim = 2  size = 6


`shape` est un attribut, pas une méthode : pas de parenthèses. C'est la
distinction du notebook 09.

`size` donne le nombre total d'éléments, soit le produit des dimensions.

### Le piège central : (3,) contre (3, 1)

C'est la source de la moitié des erreurs de dimension en machine learning.

In [65]:
v = np.array([1, 2, 3])
colonne = v.reshape(3, 1)

print('v :', v.shape, ' ndim =', v.ndim)
print(v)
print()
print('colonne :', colonne.shape, ' ndim =', colonne.ndim)
print(colonne)

v : (3,)  ndim = 1
[1 2 3]

colonne : (3, 1)  ndim = 2
[[1]
 [2]
 [3]]


Les deux contiennent les mêmes valeurs, mais ce ne sont pas les mêmes objets.

`(3,)` est un vecteur à une dimension. La virgule solitaire dans le tuple
n'est pas une coquille : c'est ce qui en fait un tuple d'un élément, comme vu au
notebook 04.

`(3, 1)` est une matrice de trois lignes et une colonne, donc à deux
dimensions.

La conséquence pratique se voit dès qu'on calcule :

In [66]:
print('v + v.T :', (v + v.T).shape) # reste (3,)
print('colonne + colonne.T :', (colonne + colonne.T).shape) # devient (3, 3) !

v + v.T : (3,)
colonne + colonne.T : (3, 3)


Additionner un vecteur colonne et sa transposée produit une matrice 3×3, par
broadcasting le sujet du notebook 13.

C'est exactement pour cette raison que scikit-learn exige souvent un
`reshape(-1, 1)` sur une variable unique : ses modèles attendent une matrice à
deux dimensions, une ligne par individu.

Le réflexe : avant toute opération, afficher la forme.

```python
print(X.shape, y.shape)
```

C'est ce qui aurait évité les erreurs rencontrées avec `x0` et `X[:, 2]` dans le
notebook SciPy.

## 3. Les constructeurs

Plutôt que de partir d'une liste, on crée souvent un tableau directement à la
bonne forme.

In [67]:
print('zeros(3) :', np.zeros(3))
print()
print('zeros((3, 2)) :')
print(np.zeros((3, 2)))

zeros(3) : [0. 0. 0.]

zeros((3, 2)) :
[[0. 0.]
 [0. 0.]
 [0. 0.]]


Attention à la double parenthèse. `np.zeros(3, 2)` échoue, parce que le second
argument est réservé au type.

In [68]:
try:
    np.zeros(3, 2)
except TypeError as e:
    print('TypeError :', str(e)[:70])

print()
print('correct :', np.zeros((3, 2)).shape)

TypeError : Cannot interpret '2' as a data type

correct : (3, 2)


La forme se donne en un seul argument, sous forme de tuple. C'est cohérent
avec `shape`, qui renvoie un tuple.

In [69]:
print('ones((3, 4)) :')
print(np.ones((3, 4)))

print()
print('full((2, 3), 7) :')
print(np.full((2, 3), 7))

print()
print('eye(4) : la matrice identité')
print(np.eye(4))

ones((3, 4)) :
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]

full((2, 3), 7) :
[[7 7 7]
 [7 7 7]]

eye(4) : la matrice identité
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


`eye` construit la matrice identité des 1 sur la diagonale, des 0 ailleurs.
C'est exactement ce qu'on avait écrit à la main en compréhension au notebook 06.

Deux autres constructeurs utiles :

In [70]:
print('empty : non initialisé, contenu quelconque')
print(np.empty((2, 2)))

print()
print('zeros_like : même forme qu\'un autre tableau')
M = np.array([[1, 2, 3], [4, 5, 6]])
print(np.zeros_like(M))

empty : non initialisé, contenu quelconque
[[1. 1.]
 [1. 1.]]

zeros_like : même forme qu'un autre tableau
[[0 0 0]
 [0 0 0]]


`empty` est plus rapide que `zeros` puisqu'il n'écrit rien, mais son contenu est
imprévisible. À n'utiliser que si l'on remplit tout ensuite.

`zeros_like`, `ones_like` et `full_like` évitent de recopier une forme à la main utile quand elle vient d'un calcul.

## 4. linspace et arange

Deux façons de générer une suite régulière, avec une différence importante.

In [71]:
print('linspace(0, 10, 5) :', np.linspace(0, 10, 5))
print('arange(0, 10, 2)   :', np.arange(0, 10, 2))

linspace(0, 10, 5) : [ 0.   2.5  5.   7.5 10. ]
arange(0, 10, 2)   : [0 2 4 6 8]


| | Troisième argument | Borne finale |
|---|---|---|
| `linspace(début, fin, n)` | le **nombre de points** | **incluse** |
| `arange(début, fin, pas)` | le **pas** | **exclue** |

`linspace(0, 10, 5)` donne cinq valeurs et se termine à 10. `arange(0, 10, 2)`
avance de 2 en 2 et s'arrête avant 10.

L'exclusion de la borne dans `arange` est cohérente avec `range` du notebook 03.

### Compter les points de linspace

In [72]:
print('linspace(0, 10, 20) :', np.linspace(0, 10, 20)[:4], '... pas =',
      round(np.linspace(0, 10, 20)[1], 4))
print('linspace(0, 10, 21) :', np.linspace(0, 10, 21)[:4], '... pas =',
      round(np.linspace(0, 10, 21)[1], 4))

linspace(0, 10, 20) : [0.         0.52631579 1.05263158 1.57894737] ... pas = 0.5263
linspace(0, 10, 21) : [0.  0.5 1.  1.5] ... pas = 0.5


Avec 21 points, le pas tombe rond à 0,5 ; avec 20, il vaut 0,526. La règle : pour
un pas de `p` sur un intervalle de longueur `L`, il faut `L/p + 1` points.

C'est le fameux problème des piquets de clôture un de plus que d'intervalles.

### Le piège d'arange avec des décimaux

`arange` accepte un pas flottant, mais le résultat n'est pas fiable.

In [73]:
a = np.arange(0, 1, 0.1)
print(a)
print('nombre de points :', a.size, ' dernier :', a[-1])

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
nombre de points : 10  dernier : 0.9


Ici tout va bien, mais l'accumulation d'erreurs d'arrondi peut faire apparaître
ou disparaître un point selon les valeurs. C'est le problème des flottants du
notebook 02, appliqué à une addition répétée.

La documentation NumPy recommande explicitement `linspace` dès que le pas n'est
pas entier. Avec `linspace`, on contrôle le nombre de points, qui lui est
exact.

In [74]:
print('arange :', np.arange(0, 1, 0.1).size, 'points, dernier =',
      np.arange(0, 1, 0.1)[-1])
print('linspace :', np.linspace(0, 1, 11).size, 'points, dernier =',
      np.linspace(0, 1, 11)[-1])

arange : 10 points, dernier = 0.9
linspace : 11 points, dernier = 1.0


## 5. Les types de données

Contrairement à une liste, un tableau NumPy a un seul type pour tous ses
éléments. C'est ce qui permet le rangement en bloc continu et la vitesse.

In [75]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)
print(np.array([True, False]).dtype)
print(np.array(['a', 'bb']).dtype)

int64
float64
bool
<U2


Un seul flottant suffit à convertir tout le tableau. NumPy choisit le type le
plus général capable de contenir toutes les valeurs.

On peut l'imposer avec `dtype` :

In [76]:
print(np.arange(0, 10, 2)) # int par défaut
print(np.arange(0, 10, 2, dtype=float)) # forcé en flottant
print(np.arange(0, 10, 2, dtype=np.float32))

[0 2 4 6 8]
[0. 2. 4. 6. 8.]
[0. 2. 4. 6. 8.]


### Pourquoi choisir un type

Le type détermine la place occupée et la précision.

In [77]:
n = 1_000_000

for t in [np.int8, np.int32, np.int64, np.float16, np.float32, np.float64]:
    tab = np.ones(n, dtype=t)
    print(f'{str(np.dtype(t)):<10} {tab.nbytes / 1e6:>6.1f} Mo')

int8          1.0 Mo
int32         4.0 Mo
int64         8.0 Mo
float16       2.0 Mo
float32       4.0 Mo
float64       8.0 Mo


Un tableau d'un million de valeurs occupe 1 Mo en `int8` contre 8 Mo en
`float64`. Sur des images ou de gros jeux de données, le choix compte.

`float64` est le type par défaut. Il correspond au `float` de Python et offre
environ 15 chiffres significatifs.

### Le piège du dépassement

Un type trop petit ne prévient pas.

In [78]:
print('Maximum de float16 :', np.finfo(np.float16).max)
print()
print(np.arange(0, 100000, 10000, dtype=np.float16))

Maximum de float16 : 65500.0

[    0. 10000. 20000. 30000. 40000. 49984. 60000.    inf    inf    inf]


Les dernières valeurs deviennent `inf` : elles dépassent la capacité du type.
Aucune erreur, juste des infinis qui contamineront tous les calculs suivants.

Même chose avec les entiers, en pire la valeur repart dans le négatif :

In [79]:
petit = np.array([127], dtype=np.int8)
print('127 en int8 :', petit)
print('127 + 1 :', (petit + np.int8(1)))

127 en int8 : [127]
127 + 1 : [-128]


`int8` va de -128 à 127. Ajouter 1 à 127 donne -128, par débordement circulaire.

C'est un piège classique en traitement d'images, où les pixels sont souvent en
`uint8` de 0 à 255. Additionner deux images sans conversion produit des zones
sombres là où l'on attendait du blanc.

La parade : convertir avant de calculer.

```python
image.astype(np.float32) + autre
```

## 6. L'aléatoire

`np.random` génère des tableaux de valeurs aléatoires.

In [80]:
np.random.seed(0)
print('randn : loi normale, centrée en 0')
print(np.random.randn(2, 3).round(3))

print()
print('rand : loi uniforme entre 0 et 1')
print(np.random.rand(2, 3).round(3))

print()
print('randint : entiers')
print(np.random.randint(0, 10, (2, 3)))

randn : loi normale, centrée en 0
[[ 1.764  0.4    0.979]
 [ 2.241  1.868 -0.977]]

rand : loi uniforme entre 0 et 1
[[0.438 0.892 0.964]
 [0.383 0.792 0.529]]

randint : entiers
[[8 1 5]
 [9 8 9]]


Le `n` de `randn` signifie *normal*. La différence avec `rand` compte :

`rand` tire uniformément entre 0 et 1 toutes les valeurs sont également
probables.

`randn` tire selon une loi normale centrée en 0, d'écart-type 1 les valeurs
proches de 0 sont plus fréquentes, et des valeurs négatives apparaissent.

In [81]:
np.random.seed(0)
print('rand  : min =', np.random.rand(1000).min().round(3),
      ' max =', np.random.rand(1000).max().round(3))

np.random.seed(0)
echantillon = np.random.randn(10000)
print('randn : min =', echantillon.min().round(2),
      ' max =', echantillon.max().round(2),
      ' moyenne =', echantillon.mean().round(3))

rand  : min = 0.001  max = 0.999
randn : min = -3.74  max = 3.8  moyenne = -0.018


### La reproductibilité

`np.random.seed` fixe le générateur, comme `random.seed` au notebook 08.

In [82]:
np.random.seed(0)
print(np.random.randn(3).round(3))

np.random.seed(0)
print(np.random.randn(3).round(3)) # identique

print(np.random.randn(3).round(3)) # différent

[1.764 0.4   0.979]
[1.764 0.4   0.979]
[ 2.241  1.868 -0.977]


Point important : `random.seed` et `np.random.seed` sont indépendants.
Fixer l'un ne fixe pas l'autre. Un notebook qui utilise les deux doit fixer les
deux.

Et la graine agit sur toute la suite : réexécuter une cellule du milieu ne donne
pas les mêmes valeurs qu'au premier passage. Pour un notebook publié, mets le
`seed` dans chaque cellule qui tire de l'aléatoire.

La documentation recommande aujourd'hui une approche plus moderne, avec un
générateur explicite :

```python
rng = np.random.default_rng(0)
rng.normal(size=(2, 3))
rng.integers(0, 10, size=5)
```

L'avantage est que chaque générateur est indépendant : plusieurs parties d'un
programme peuvent avoir leur propre suite sans interférer.

## 7. Assembler des tableaux

In [83]:
A = np.zeros((3, 2))
B = np.ones((3, 2))

print('hstack : côte à côte, horizontalement')
print(np.hstack((A, B)))
print('forme :', np.hstack((A, B)).shape)

hstack : côte à côte, horizontalement
[[0. 0. 1. 1.]
 [0. 0. 1. 1.]
 [0. 0. 1. 1.]]
forme : (3, 4)


In [84]:
print('vstack : empilés verticalement')
print(np.vstack((A, B)))
print('forme :', np.vstack((A, B)).shape)

vstack : empilés verticalement
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]
forme : (6, 2)


Les tableaux à assembler se passent dans un tuple, d'où la double
parenthèse. `np.hstack(A, B)` échoue.

| Fonction | Effet | Forme obtenue à partir de (3,2) et (3,2) |
|---|---|---|
| `hstack` | colle horizontalement | (3, 4) |
| `vstack` | empile verticalement | (6, 2) |

### concatenate : la version générale

`hstack` et `vstack` sont des raccourcis de `concatenate`, qui prend un argument
`axis`.

In [85]:
print('axis=0, comme vstack :', np.concatenate((A, B), axis=0).shape)
print('axis=1, comme hstack :', np.concatenate((A, B), axis=1).shape)

axis=0, comme vstack : (6, 2)
axis=1, comme hstack : (3, 4)


L'`axis` est une notion centrale en NumPy et en Pandas, et elle mérite d'être
comprise une bonne fois.

`axis=0` est l'axe des lignes, celui qui grandit quand on empile vers le bas.
`axis=1` est l'axe des colonnes, qui grandit vers la droite.

Une façon de retenir : `axis=i` désigne la position `i` dans le tuple `shape`.
Concaténer sur `axis=0` fait grandir `shape[0]`.

C'est le même `axis` qu'on retrouvera partout :

```python
tableau.sum(axis=0) # somme de chaque colonne
tableau.sum(axis=1) # somme de chaque ligne
data.drop(columns=[...]) # équivalent de axis=1 en Pandas
```

Pour plus de deux dimensions, `concatenate` est indispensable `hstack` et
`vstack` deviennent ambigus.

### Les formes doivent être compatibles

In [86]:
try:
    np.vstack((np.zeros((3, 2)), np.ones((3, 5))))
except ValueError as e:
    print('ValueError :', str(e)[:100])

ValueError : all the input array dimensions except for the concatenation axis must match exactly, but along dimen


Pour empiler verticalement, il faut le même nombre de colonnes. Pour coller
horizontalement, le même nombre de lignes.

## 8. Changer la forme

`reshape` réorganise les éléments dans une nouvelle forme, sans en changer le
nombre.

In [87]:
d = np.arange(12)
print(d)
print()
print(d.reshape(3, 4))
print()
print(d.reshape(4, 3))

[ 0  1  2  3  4  5  6  7  8  9 10 11]

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


### Le piège : reshape ne modifie pas

C'est une méthode qui renvoie un nouveau tableau, comme `sorted` du notebook
04.

In [88]:
d = np.arange(12)

d.reshape(3, 4) # le résultat est jeté
print('forme après :', d.shape)

d = d.reshape(3, 4) # correct
print('forme après réassignation :', d.shape)

forme après : (12,)
forme après réassignation : (3, 4)


Le premier `reshape` calcule un tableau et le perd faute de destination. Même
mécanisme que `float(x)` au notebook 07.

### La taille doit correspondre

In [89]:
d = np.arange(12)

try:
    d.reshape(4, 4)
except ValueError as e:
    print('ValueError :', e)

print('12 éléments : 3x4 ou 4x3 ou 2x6, mais pas 4x4')

ValueError : cannot reshape array of size 12 into shape (4,4)
12 éléments : 3x4 ou 4x3 ou 2x6, mais pas 4x4


Le produit des dimensions doit égaler `size`. Douze éléments ne rentrent pas dans
une grille 4×4, qui en demande seize.

### Le -1 magique

Un `-1` dans `reshape` laisse NumPy calculer la dimension manquante.

In [90]:
d = np.arange(12)

print('reshape(3, -1) :', d.reshape(3, -1).shape)
print('reshape(-1, 2) :', d.reshape(-1, 2).shape)
print('reshape(-1, 1) :', d.reshape(-1, 1).shape)

reshape(3, -1) : (3, 4)
reshape(-1, 2) : (6, 2)
reshape(-1, 1) : (12, 1)


`reshape(-1, 1)` est l'idiome pour transformer un vecteur en colonne, sans avoir
à connaître sa longueur. C'est ce que réclame scikit-learn pour une variable
unique.

On ne peut mettre qu'un seul `-1` : avec deux inconnues, le calcul est
impossible.

### squeeze : retirer les dimensions inutiles

In [91]:
colonne = np.array([[1], [2], [3]])
print('avant :', colonne.shape)
print('squeeze :', colonne.squeeze().shape)

profond = np.zeros((1, 3, 1))
print()
print('avant :', profond.shape)
print('squeeze :', profond.squeeze().shape)

avant : (3, 1)
squeeze : (3,)

avant : (1, 3, 1)
squeeze : (3,)


`squeeze` supprime toutes les dimensions de taille 1. Utile pour repasser d'une
matrice colonne à un vecteur simple.

Comme `reshape`, elle renvoie un nouveau tableau et ne modifie pas l'original.

### ravel et flatten : tout aplatir

In [92]:
d = np.arange(12).reshape(3, 4)

print('ravel :', d.ravel())
print('flatten :', d.flatten())
print('formes identiques :', d.ravel().shape == d.flatten().shape)

ravel : [ 0  1  2  3  4  5  6  7  8  9 10 11]
flatten : [ 0  1  2  3  4  5  6  7  8  9 10 11]
formes identiques : True


Résultat identique, mais comportement différent, et c'est le troisième piège
annoncé.

In [93]:
d = np.zeros((3, 4))

vue = d.ravel()
vue[0] = 99
print('après modification du ravel   :', d[0, 0])

d = np.zeros((3, 4))
copie = d.flatten()
copie[0] = 99
print('après modification du flatten :', d[0, 0])

après modification du ravel   : 99.0
après modification du flatten : 0.0


`ravel` renvoie une vue : elle partage la mémoire du tableau d'origine.
Modifier l'une modifie l'autre.

`flatten` renvoie une copie indépendante.

C'est le partage d'objets rencontré depuis le notebook 02, sous une forme
propre à NumPy. La différence est que NumPy le fait pour la performance : une
vue ne coûte rien, une copie duplique les données.

La règle : `ravel` si on veut juste lire, `flatten` si on va modifier.

Pour savoir si un tableau est une vue :

In [94]:
d = np.zeros((3, 4))
print('ravel partage :', d.ravel().base is d)
print('flatten partage :', d.flatten().base is d)

ravel partage : True
flatten partage : False


L'attribut `base` désigne le tableau d'origine quand il s'agit d'une vue, et vaut
`None` pour un tableau indépendant.

Le slicing aussi renvoie des vues, contrairement aux listes Python :

In [95]:
liste = [1, 2, 3, 4]
morceau_liste = liste[1:3]
morceau_liste[0] = 99
print('liste Python :', liste, '-> inchangée, slicing = copie')

tab = np.array([1, 2, 3, 4])
morceau_tab = tab[1:3]
morceau_tab[0] = 99
print('tableau NumPy:', tab, '-> modifiée, slicing = vue')

liste Python : [1, 2, 3, 4] -> inchangée, slicing = copie
tableau NumPy: [ 1 99  3  4] -> modifiée, slicing = vue


C'est une différence majeure avec les listes, et une source de bugs quand on
passe de l'une à l'autre. Le sujet est repris au notebook 11.

## 9. Pourquoi NumPy

Une démonstration chiffrée, qui justifie tout ce notebook.

In [96]:
import time

n = 1_000_000
liste = list(range(n))
tableau = np.arange(n)

debut = time.time()
resultat_py = [x**2 for x in liste]
temps_py = time.time() - debut

debut = time.time()
resultat_np = tableau ** 2
temps_np = time.time() - debut

print(f'Compréhension Python : {temps_py:.4f} s')
print(f'NumPy : {temps_np:.5f} s')
print(f'Rapport : {temps_py/temps_np:.0f}x')

Compréhension Python : 0.0538 s
NumPy : 0.00273 s
Rapport : 20x


Et la mémoire :

In [97]:
import sys
print('Liste Python :', sys.getsizeof(liste) / 1e6, 'Mo (hors objets pointés)')
print('Tableau NumPy:', tableau.nbytes / 1e6, 'Mo')

Liste Python : 8.000056 Mo (hors objets pointés)
Tableau NumPy: 8.0 Mo


La liste est en réalité bien plus lourde que ce que `getsizeof` indique : elle ne
compte que les pointeurs, pas les objets entiers eux-mêmes, qui font 28 octets
chacun.

Les trois raisons de l'écart.

Un tableau NumPy range ses éléments dans un bloc mémoire continu, ce qui
permet au processeur de les charger par paquets.

Le type unique évite de vérifier la nature de chaque élément à chaque
opération.

Et les calculs sont effectués en C compilé, pas en Python interprété.

C'est ce qu'on appelle la vectorisation : écrire `tableau 2` plutôt qu'une
boucle. La règle générale en NumPy est simple si tu écris une boucle `for` sur
un tableau, il existe presque toujours une meilleure façon.

## 10. Mémo

### Créer

| Fonction | Résultat |
|---|---|
| `np.array([1, 2, 3])` | depuis une liste |
| `np.zeros((3, 2))` | que des zéros |
| `np.ones((3, 2))` | que des uns |
| `np.full((3, 2), 7)` | une valeur constante |
| `np.eye(4)` | matrice identité |
| `np.linspace(0, 10, 5)` | n points, borne incluse |
| `np.arange(0, 10, 2)` | pas fixe, borne exclue |
| `np.random.randn(3, 4)` | loi normale |
| `np.random.rand(3, 4)` | loi uniforme |

### Inspecter

| Attribut | Signification |
|---|---|
| `.shape` | la forme, un tuple |
| `.ndim` | le nombre de dimensions |
| `.size` | le nombre total d'éléments |
| `.dtype` | le type des éléments |
| `.nbytes` | la place occupée |

### Transformer

| Méthode | Effet |
|---|---|
| `.reshape(3, 4)` | nouvelle forme |
| `.reshape(-1, 1)` | en colonne, longueur déduite |
| `.squeeze()` | retire les dimensions de taille 1 |
| `.ravel()` | aplatit, renvoie une vue |
| `.flatten()` | aplatit, renvoie une copie |
| `.T` | transposée |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| `np.array(1, 2, 3)` | `TypeError`, il faut une séquence |
| `np.zeros(3, 2)` | `TypeError`, la forme est un tuple |
| `(3,)` contre `(3, 1)` | dimensions différentes, broadcasting différent |
| `d.reshape(3, 4)` sans réassigner | le tableau reste inchangé |
| `arange` avec un pas décimal | nombre de points imprévisible |
| `dtype` trop petit | dépassement silencieux, `inf` ou valeur négative |
| `ravel` modifié | modifie le tableau d'origine |
| Slicing NumPy | renvoie une vue, pas une copie |

## 11. Exercices

**Exercice 1**

Écris une fonction `initialisation(m, n)` qui renvoie une matrice de forme
`(m, n+1)` : `n` colonnes de valeurs aléatoires normales, et une dernière
colonne remplie de 1.

C'est exactement ce qu'on fait en régression linéaire pour intégrer le terme
constant. Vérifie la forme et le contenu de la dernière colonne.

**Exercice 2**

Crée un tableau de 20 valeurs entre 0 et 1 de trois façons : `linspace`,
`arange`, et une compréhension convertie en tableau. Compare les formes, les
premières et dernières valeurs, et les temps d'exécution sur un million de
points.

Laquelle utiliserais-tu, et pourquoi ?

**Exercice 3**

Cette fonction contient deux pièges de ce notebook :

```python
def normaliser(tableau):
    plat = tableau.ravel()
    plat -= plat.min()
    plat /= plat.max()
    tableau.reshape(-1, 1)
    return plat
```

Teste-la sur un tableau 3×4, puis affiche le tableau d'origine. Explique les deux
problèmes, puis corrige la fonction.

## Pour continuer

Le notebook suivant porte sur l'indexing et le slicing : atteindre une portion
précise d'un tableau, et le masque booléen celui qu'on utilisait déjà en Pandas
avec `data[data['age'] < 18]`.

Puis viendront les mathématiques et les statistiques, et enfin le broadcasting,
qui explique pourquoi `(3,)` et `(3, 1)` ne se comportent pas pareil.

In [98]:
# Exercice 1

def initialisation(m, n, graine=0):
    """Crée une matrice (m, n+1) : n colonnes aléatoires et une colonne de 1.
    La dernière colonne sert de terme constant en régression linéaire.
    """
    rng = np.random.default_rng(graine)
    X = rng.normal(size=(m, n)) # (m, n) valeurs normales
    biais = np.ones((m, 1)) # (m, 1) colonne de 1
    return np.hstack((X, biais))

In [116]:
X = initialisation(4, 3)

print(X.round(2))
print()
print('forme attendue (4, 4) :', X.shape)
print('dernière colonne :', X[:, -1])
print('que des 1 ?', (X[:, -1] == 1).all())

[[ 0.13 -0.13  0.64  1.  ]
 [ 0.1  -0.54  0.36  1.  ]
 [ 1.3   0.95 -0.7   1.  ]
 [-1.27 -0.62  0.04  1.  ]]

forme attendue (4, 4) : (4, 4)
dernière colonne : [1. 1. 1. 1.]
que des 1 ? True


In [118]:
try:
    np.hstack((np.zeros((4, 3)), np.ones(4)))
except ValueError as e:
    print('ValueError :', str(e)[:90])

ValueError : all the input arrays must have same number of dimensions, but the array at index 0 has 2 d


In [120]:
theta = np.random.randn(4, 1) # 3 coefficients + le biais
predictions = X.dot(theta)
print('prédictions :', predictions.shape)

prédictions : (4, 1)


In [102]:
# Exerice 2

In [122]:
a = np.linspace(0, 1, 20)
b = np.arange(0, 1, 0.05)
c = np.array([i / 19 for i in range(20)])

print('formes :', a.shape, b.shape, c.shape)
print('premières:', a[0], b[0], c[0])
print('dernières:', a[-1], b[-1], c[-1])

formes : (20,) (20,) (20,)
premières: 0.0 0.0 0.0
dernières: 1.0 0.9500000000000001 1.0


In [123]:
import time

N = 1_000_000

debut = time.time(); np.linspace(0, 1, N);              t_lin = time.time() - debut
debut = time.time(); np.arange(0, 1, 1/N);              t_ara = time.time() - debut
debut = time.time(); np.array([i/(N-1) for i in range(N)]); t_comp = time.time() - debut

print(f'linspace : {t_lin:.5f} s')
print(f'arange : {t_ara:.5f} s')
print(f'compréhension : {t_comp:.4f} s')
print(f'rapport : {t_comp/t_lin:.0f}x')

linspace : 0.00274 s
arange : 0.00132 s
compréhension : 0.1206 s
rapport : 44x


In [124]:
import time

N = 1_000_000

debut = time.time(); np.linspace(0, 1, N)
t_lin = time.time() - debut
debut = time.time(); np.arange(0, 1, 1/N)
t_ara = time.time() - debut
debut = time.time(); np.array([i/(N-1) for i in range(N)])
t_comp = time.time() - debut

print(f'linspace : {t_lin:.5f} s')
print(f'arange : {t_ara:.5f} s')
print(f'compréhension : {t_comp:.4f} s')
print(f'rapport : {t_comp/t_lin:.0f}x')

linspace : 0.00752 s
arange : 0.00765 s
compréhension : 0.2769 s
rapport : 37x


In [125]:
# Exerice 3

In [126]:
def normaliser(tableau):
    plat = tableau.ravel()
    plat -= plat.min()
    plat /= plat.max()
    tableau.reshape(-1, 1)
    return plat

In [108]:
d = np.arange(12, dtype=float).reshape(3, 4)
original = d.copy()

resultat = normaliser(d)

print('résultat :', resultat.round(2))
print()
print('tableau d\'origine :')
print(d.round(2))
print()
print('modifié ?', not np.array_equal(d, original))

résultat : [0.   0.09 0.18 0.27 0.36 0.45 0.55 0.64 0.73 0.82 0.91 1.  ]

tableau d'origine :
[[0.   0.09 0.18 0.27]
 [0.36 0.45 0.55 0.64]
 [0.73 0.82 0.91 1.  ]]

modifié ? True


In [109]:
d = np.arange(12, dtype=float).reshape(3, 4)
print('ravel est une vue :', d.ravel().base is d)

ravel est une vue : False


In [110]:
d = np.arange(12, dtype=float).reshape(3, 4)
print('ravel est une vue :', d.ravel().base is d)

ravel est une vue : False


In [111]:
tableau.reshape(-1, 1)

array([[     0],
       [     1],
       [     2],
       ...,
       [999997],
       [999998],
       [999999]])

In [112]:
e = np.arange(12).reshape(3, 4) # entiers, pas de dtype=float
try:
    normaliser(e)
except Exception as ex:
    print(type(ex).__name__, ':', str(ex)[:80])

UFuncTypeError : Cannot cast ufunc 'divide' output from dtype('float64') to dtype('int64') with c


In [113]:
def normaliser(tableau):
    """Ramène les valeurs entre 0 et 1, sans modifier le tableau d'origine.

    Retourne un tableau de même forme que l'entrée.
    """
    resultat = tableau.astype(float) # copie et conversion en flottant

    minimum = resultat.min()
    etendue = resultat.max() - minimum

    if etendue == 0: # toutes les valeurs identiques
        return np.zeros_like(resultat)

    return (resultat - minimum) / etendue

In [114]:
d = np.arange(12).reshape(3, 4)
original = d.copy()

resultat = normaliser(d)

print(resultat.round(2))
print()
print('origine intacte :', np.array_equal(d, original))
print('forme conservée :', resultat.shape)
print('min et max :', resultat.min(), resultat.max())

[[0.   0.09 0.18 0.27]
 [0.36 0.45 0.55 0.64]
 [0.73 0.82 0.91 1.  ]]

origine intacte : True
forme conservée : (3, 4)
min et max : 0.0 1.0


In [115]:
constant = np.full((2, 3), 5)
print(normaliser(constant))

[[0. 0. 0.]
 [0. 0. 0.]]
